In [997]:
# Import necessary packages
import torch
import torch.nn as nn
from torchvision.transforms import v2
from PIL import Image

# Import packages for training and semi-supervised learning
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision.datasets import DatasetFolder

In [998]:
# Define transforms (data augmentation) for training data
train_tfm = v2.Compose([
    # Data augmentation
    v2.RandomAffine(degrees=60), # Rotation
    v2.RandomHorizontalFlip(p=0.5),
    v2.AutoAugment(), # Use ImageNet data augmentation policies
    v2.Resize((128, 128)), # Resize images into a fixed shape, which is width = hight = 128
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# Do not need augmentations in test and validations
test_tfm = v2.Compose([
    v2.Resize((128, 128)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

In [999]:
def iamge_loader(path):
    return Image.open(path)

In [1000]:
batch_size = 128

# Construct datasets
# The parameter transform tells the DatasetFolder how to read the data
train_set = DatasetFolder("food-11/training/labeled", loader=iamge_loader, extensions="jpg", transform=train_tfm)
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=iamge_loader, extensions="jpg", transform=train_tfm)
validation_set = DatasetFolder("food-11/validation", loader=iamge_loader, extensions="jpg", transform=test_tfm)
test_set = DatasetFolder("food-11/testing", loader=iamge_loader, extensions="jpg", transform=test_tfm)

# Construct dataloaders
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
validation_loader = DataLoader(validation_set, batch_size=batch_size, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [1001]:
# Define model architecture
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # input image size: [3, 128, 128]
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),
            
            nn.Conv2d(64, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),
            
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(4, 4, 0)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Linear(256, 11)
        )
        
    def forward(self, x):
        # Extract features in images by convolutional layers
        x = self.cnn_layers(x)
        
        # The extracted feature map must be flatten before going to fully-connected layers
        x = x.flatten(1)
        
        # Use fully-connected layers to learn and obtian final logits
        output = self.fc_layers(x)
        
        return output

In [1002]:
# Define a pseudo dataset
class PseudoDataset(Dataset):
    def __init__(self, dataset, indices, pseudo_labels):
        self.dataset = dataset
        self.indices = indices
        self.pseudo_labels = pseudo_labels
        
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        img, _ = self.dataset[self.indices[idx]]
        return img, self.pseudo_labels[idx]

In [1003]:
# Define a function to get pseudo labels for unlabeled data, which is necessary in semi-supervised learning
def get_pseudo_labels(dataset, model, threshold=0.65):
    # This function returns an instance of torchvision.datasets.DatasetFolder containing images whose prediction confidences exceed a given threshold
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Construct a dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval() # Set the model as evaluation mode
    softmax = nn.Softmax(dim=-1)
    
    selected_indices = []
    pseudo_labels = []
    start_idx = 0 # start index of current batch
    
    for batch in dataloader:
        imgs, _ = batch
        with torch.no_grad():
            logits = model(imgs.to(device))
        probs = softmax(logits)
        max_probs, preds = torch.max(probs, dim=1)
        # for i in range(len(max_probs)):
        #     if max_probs[i] >= threshold:
        #         selected_indices.append(start_idx + i)
        #         pseudo_labels.append(preds[i].item())
        mask = max_probs >= threshold
        selected_indices.extend((torch.where(mask)[0] + start_idx).cpu().tolist())
        
        pseudo_labels.extend(preds[mask].cpu().tolist())
        
        # Update start index of next batch
        start_idx += len(imgs)
                
    model.train() # Set the model back to training mode
                
    return PseudoDataset(dataset, selected_indices, pseudo_labels)

In [1004]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Classifier().to(device)
model.device = device
model_path = "./model.ckpt"

# Use cross-entropy as loss
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-5)

num_epochs = 200

early_stop = 50

do_semi = True
pseudo_label_threshold = 0.95

In [1005]:
# Train
min_acc = 0
early_stop_count = 0

for epoch in range(num_epochs):
    if do_semi & ((epoch + 1) % 10 == 0):
        # Get a pseudo dataset with pseudo labels and concat it with the training set
        pseudo_set = get_pseudo_labels(unlabeled_set, model, threshold=pseudo_label_threshold)
        print(f"Get {len(pseudo_set)} samples with pseudo label")
        concated_set = ConcatDataset([train_set, pseudo_set])
        train_loader = DataLoader(concated_set, batch_size=batch_size, shuffle=True, pin_memory=True)
        
    model.train()
    
    train_loss = []
    train_accs = []
    
    for batch in train_loader:
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        optimizer.zero_grad() # Clear previous gradients
        loss.backward() # Calculate gradients
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=10) # Clip the gradient norms for stabel training, prevent exploding gradients
        optimizer.step() # Update weights
        
        acc = (logits.argmax(dim=-1) == labels).float().mean()
        train_loss.append(loss.item())
        train_accs.append(acc)
        
    avg_train_loss = sum(train_loss) / len(train_loss)
    avg_train_acc = sum(train_accs) / len(train_accs)
    
    print(f"[Train | {epoch + 1:03d}/{num_epochs:03d}] loss = {avg_train_loss:.5f}, acc = {avg_train_acc:.5f}")
    
    # Validation
    model.eval()
    
    validation_loss = []
    validation_accs = []
    
    for batch in validation_loader:
        imgs, labels = batch
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            logits = model(imgs)
            loss = criterion(logits, labels)
            
            acc = (logits.argmax(dim=-1) == labels).float().mean()
            validation_loss.append(loss.item())
            validation_accs.append(acc)
            
    avg_validation_loss = sum(validation_loss) / len(validation_loss)
    avg_validation_acc = sum(validation_accs) / len(validation_accs)
    
    print(f"[Validation | {epoch + 1:03d}/{num_epochs:03d}] loss = {avg_validation_loss:.5f}, acc = {avg_validation_acc:.5f}")
    
    if avg_validation_acc > min_acc:
        min_acc = avg_validation_acc
        torch.save(model.state_dict(), model_path)
        print(f"Save model with acc: {min_acc:.3f}")
        early_stop_count = 0
    else:
        early_stop_count += 1
    
    if early_stop_count >= early_stop:
        print(f"Finish training at the end of epoch {epoch + 1}, min acc is {min_acc:.3f}")
        break

[Train | 001/200] loss = 2.63398, acc = 0.11312
[Validation | 001/200] loss = 2.36545, acc = 0.10807
Save model with acc: 0.108
[Train | 002/200] loss = 2.34973, acc = 0.15187
[Validation | 002/200] loss = 2.35353, acc = 0.14453
Save model with acc: 0.145
[Train | 003/200] loss = 2.25305, acc = 0.19969
[Validation | 003/200] loss = 2.27606, acc = 0.19245
Save model with acc: 0.192
[Train | 004/200] loss = 2.20193, acc = 0.21406
[Validation | 004/200] loss = 2.09284, acc = 0.25573
Save model with acc: 0.256
[Train | 005/200] loss = 2.15903, acc = 0.23719
[Validation | 005/200] loss = 1.86624, acc = 0.37943
Save model with acc: 0.379
[Train | 006/200] loss = 2.09167, acc = 0.27531
[Validation | 006/200] loss = 1.83587, acc = 0.36823
[Train | 007/200] loss = 2.04937, acc = 0.27656
[Validation | 007/200] loss = 1.74569, acc = 0.41120
Save model with acc: 0.411
[Train | 008/200] loss = 2.01485, acc = 0.29063
[Validation | 008/200] loss = 2.16126, acc = 0.31328
[Train | 009/200] loss = 1.983

In [1008]:
# Test
model = Classifier().to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

predictions = []

for batch in test_loader:
    imgs, _ = batch
    imgs = imgs.to(device)
    with torch.no_grad():
        logits = model(imgs)
        
    # Take the class with greatest logit as prediction and record it
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

In [1009]:
# Save predictions into a file
with open("prediction.csv", 'w') as f:
    f.write("Id,Category\n")
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")